## Can gpt2_small store facts already or do I need to work with a different model?

- Load weights into gpt2_model and run prompts that test for evidence of stored facts
- Assess its ability to store facts
- Look online to understand the capabilities of gpt2_small
- Look online for similar assessments

In [ ]:
# Setup
import torch as t
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial

# import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.hook_points import HookPoint

# Hooking utilities
from transformer_lens import FactoredMatrix, HookedTransformer
from transformer_lens.model_bridge import TransformerBridge

In [ ]:
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)


def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(utils.to_numpy(tensor), labels={"x": xaxis, "y": yaxis}, **kwargs).show(renderer)


def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x": xaxis, "y": yaxis, "color": caxis}, **kwargs).show(renderer)

In [ ]:
t.set_grad_enabled(False)
device = t.device("mps")

gpt2_small = TransformerBridge.boot_transformers("gpt2", device=device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
END_OF_TEXT = 50256

def complete(model, prompt, max_tokens=50):
    tokens = model.tokenizer.encode(prompt)
    for n in range(max_tokens):
        logits, cache = model.run_with_cache(model.to_string(tokens))
        next_token = logits.argmax(-1).squeeze(0)[-1]
        tokens += [next_token]
        if next_token == END_OF_TEXT:
            break

    return model.to_string(tokens[1:])

print(complete(gpt2_small, "Who is the president of the United States of America?"))

Who is the president of the United States of America?


The president of the United States of America is the president of the United States of America.


The president of the United States of America is the president of the United States of America.


The president of the United States


Clearly it's going to be difficult for GPT2 to store a fact if it can't complete a sentence like this.

Let's try a simpler example and see what happens

In [ ]:
print(complete(gpt2_small, "What is the colour of the sky?"))

What is the colour of the sky?


The colour of the sky is determined by the brightness of the sun and moon. The colour of the sky is determined by the brightness of the sun and moon.


The colour of the sky is determined by the brightness of the


OK we can stop here and upgrade the model. 

In [ ]:
pythia1b = TransformerBridge.boot_transformers("EleutherAI/pythia-1b", device=device)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [ ]:
print(complete(pythia1b, "Q: What is the colour of the sky?\nA: "))

Q: What is the colour of the sky?
A: 

<|endoftext|>Q:

 is not a valid element in the document

  <div class="row">
    <div class="col-md-6">
      <div class="row">
        <div class="


It seems to want to write HTML code really badly.
I'll try another few models soon